# cv_experience_ocr_bert: etapas pesadas en Colab

Este notebook **no contiene lógica propia**: clona el repo y ejecuta los scripts de `src/`. Todo cambio de código se hace en la laptop, va a una rama y entra a `main` por PR. Aquí solo se hace `git pull`.

Las salidas pesadas (imágenes, OCR, pesos) se guardan en Google Drive mediante enlaces simbólicos, porque la máquina de Colab se borra al cerrar la sesión.

**Antes de empezar:**
1. *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4* (la descarga no la necesita, pero el OCR y BERT sí).
2. Panel izquierdo → 🔑 *Secrets* → crear `HF_TOKEN` con tu token de Hugging Face y activar *Notebook access*.

## 0. Preparación (correr siempre al abrir el notebook)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'Sin GPU'

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

REPO_URL = 'https://github.com/luyov-belli/cv_experience_ocr_bert'
REPO = '/content/cv_experience_ocr_bert'
DRIVE = '/content/drive/MyDrive/cv_experience_ocr_bert'
BRANCH = 'main'  # para probar una rama antes de fusionarla, cambia esto

if not os.path.exists(REPO):
    !git clone -q {REPO_URL} {REPO}
%cd {REPO}
!git fetch -q origin && git checkout -q {BRANCH} && git pull -q origin {BRANCH}
!git log --oneline -1

# Carpetas pesadas -> Drive (persisten entre sesiones)
for sub in ['data/raw', 'data/ocr', 'models/bert']:
    target = f'{DRIVE}/{sub}'
    os.makedirs(target, exist_ok=True)
    if not os.path.islink(sub):
        !rm -rf {sub}
        os.symlink(target, sub)
!ls -la data models

In [ ]:
# Dependencias (ver requirements-colab.txt). Se congelan en la versión que trae
# Colab solo las librerías de las que dependen sus herramientas internas:
#   pandas -> google-colab exige su versión exacta
#   numpy  -> numba y buena parte del stack de Colab
#   torch  -> build con CUDA de Colab
# pyarrow NO se congela: datasets 5.1 exige pyarrow>=24 y lo único en Colab que
# pide <24 es cudf, que este proyecto no usa.
# Si algo exigiera otra versión de las congeladas, pip falla aquí de forma
# explícita en vez de romper Colab en silencio.
import numpy, pandas, torch
keep = [f'{m.__name__}=={m.__version__.split("+")[0]}' for m in (numpy, pandas, torch)]
with open('/tmp/colab_keep.txt', 'w') as f:
    f.write('\n'.join(keep))
print('Se conservan:', keep)
!pip install -q -r requirements-colab.txt -c /tmp/colab_keep.txt

## 1. Descarga de datos (una sola vez)

Muestrea 800 CVs sin foto por ocupación (semilla 42) y guarda las imágenes en Drive. Toma del orden de 15–30 minutos. Si se corta, vuelve a correrla: el muestreo es determinista y produce la misma muestra.

In [ ]:
!python -m src.download_data

In [ ]:
# sampled_ids.csv vive en el repo (se versiona). Se copia a Drive para que
# lo bajes a la laptop y lo subas en un commit.
!cp data/sampled_ids.csv {DRIVE}/data/sampled_ids.csv
!ls {DRIVE}/data/raw | head; ls {DRIVE}/data/raw/images | wc -l